# 12. Qwen2.5-VL Streamlit 멀티모달 챗

## 학습 목표

앞 단계에서는 Notebook에서 Qwen2.5-VL의 멀티이미지와 멀티턴 대화를 구현했다.

이번 단계에서는 이를 실제 웹 애플리케이션 구조로 확장한다.

```text
사용자
  ↓
Streamlit Web UI
  ├── 이미지 업로드
  ├── 이미지 미리보기
  ├── 한국어 질문
  ├── 대화 History
  └── 대화 초기화
          ↓
      Qwen2.5-VL
          ↓
      한국어 답변
```

구현 기능은 다음과 같다.

- 단일 이미지 업로드
- 여러 이미지 업로드
- 이미지 미리보기
- 한국어 질문 입력
- Qwen2.5-VL 추론
- 멀티턴 대화
- Session State 기반 History 관리
- 새 이미지 업로드 시 대화 초기화
- 여러 이미지 비교
- 대화 초기화 버튼
- GPU / MPS / CPU 처리
- 모델 캐싱
- 오류 처리
- `app.py` 자동 생성
- Streamlit 실행 방법

# 1. Streamlit에서 중요한 점

Streamlit은 사용자가 Widget을 조작할 때 Python Script를 다시 실행한다.

따라서 일반 Python 변수만 사용하면 대화 History를 안정적으로 유지하기 어렵다.

이를 위해 다음 구조를 사용한다.

```text
st.session_state
├── messages
├── images
└── image_signature
```

`messages`에는 대화 History를 저장한다.

`images`에는 현재 대화에서 사용할 PIL Image를 저장한다.

`image_signature`는 업로드 이미지가 변경되었는지 판단하는 데 사용한다.

# 2. 모델은 매번 다시 로드하면 안 된다

Streamlit은 rerun이 발생하므로 다음과 같이 모델을 일반 함수에서 매번 로드하면 비효율적이다.

```text
사용자 질문
   ↓
Script rerun
   ↓
3B 모델 다시 로드
   ↓
질문
   ↓
Script rerun
   ↓
3B 모델 다시 로드
```

따라서 모델과 Processor처럼 재사용 가능한 Resource는 캐싱한다.

이번 예제에서는 다음 구조를 사용한다.

```python
@st.cache_resource
def load_model():
    ...
```

# 3. 프로젝트 구조

Notebook을 실행하면 같은 폴더에 다음 파일을 생성한다.

```text
qwen_vl_streamlit/
│
├── app.py
└── README.txt
```

실제 웹 애플리케이션은 `app.py`에 구현한다.

# 4. 프로젝트 폴더 생성

In [7]:
# ---------------------------------------------------------
# 운영체제에 독립적인 파일 경로 처리를 위해 Path를 불러온다.
# ---------------------------------------------------------
from pathlib import Path

# ---------------------------------------------------------
# Streamlit 프로젝트 폴더 경로를 지정한다.
# ---------------------------------------------------------
PROJECT_DIR = Path(
    "./qwen_vl_streamlit"
)

# ---------------------------------------------------------
# 프로젝트 폴더가 없으면 새로 생성한다.
# ---------------------------------------------------------
PROJECT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ---------------------------------------------------------
# Streamlit 애플리케이션 파일 경로를 지정한다.
# ---------------------------------------------------------
APP_PATH = PROJECT_DIR / "app.py"

# ---------------------------------------------------------
# 실행 안내 파일 경로를 지정한다.
# ---------------------------------------------------------
README_PATH = PROJECT_DIR / "README.txt"

# ---------------------------------------------------------
# 프로젝트 폴더의 절대 경로를 출력한다.
# ---------------------------------------------------------
print(
    "Project Directory:",
    PROJECT_DIR.resolve()
)

Project Directory: C:\sk-encoa\runpod\qwen_vl_streamlit


# 5. Streamlit 애플리케이션 전체 소스

아래 문자열이 실제 `app.py`의 전체 코드이다.

수업에서 바로 설명할 수 있도록 **실행 문장 위에 주석**을 배치했다.

In [8]:
APP_SOURCE = r''''''

# ---------------------------------------------------------
# 메모리 정리를 위해 gc를 불러온다.
# ---------------------------------------------------------
import gc

# ---------------------------------------------------------
# 업로드 파일의 변경 여부를 확인하기 위해 hashlib을 불러온다.
# ---------------------------------------------------------
import hashlib

# ---------------------------------------------------------
# 실행 시간을 측정하기 위해 time을 불러온다.
# ---------------------------------------------------------
import time

# ---------------------------------------------------------
# PyTorch를 불러온다.
# ---------------------------------------------------------
import torch

# ---------------------------------------------------------
# Streamlit Web UI를 구현하기 위해 streamlit을 불러온다.
# ---------------------------------------------------------
import streamlit as st

# ---------------------------------------------------------
# 업로드 이미지를 처리하기 위해 PIL Image를 불러온다.
# ---------------------------------------------------------
from PIL import Image

# ---------------------------------------------------------
# Qwen2.5-VL 입력 처리를 위해 AutoProcessor를 불러온다.
# ---------------------------------------------------------
from transformers import AutoProcessor

# ---------------------------------------------------------
# Qwen2.5-VL 생성 모델 클래스를 불러온다.
# ---------------------------------------------------------
from transformers import Qwen2_5_VLForConditionalGeneration


# ---------------------------------------------------------
# Streamlit 페이지의 기본 설정을 지정한다.
# ---------------------------------------------------------
st.set_page_config(
    page_title="Qwen2.5-VL Multimodal Chat",
    page_icon=None,
    layout="wide"
)


# ---------------------------------------------------------
# 애플리케이션 제목을 출력한다.
# ---------------------------------------------------------
st.title("Qwen2.5-VL 한국어 멀티모달 챗")


# ---------------------------------------------------------
# 애플리케이션 설명을 출력한다.
# ---------------------------------------------------------
st.caption(
    "이미지를 업로드하고 한국어로 질문할 수 있습니다. "
    "CPU에서는 속도를 위해 이미지와 출력 Token을 제한합니다."
)


# ---------------------------------------------------------
# 사용할 Qwen2.5-VL 모델 이름을 지정한다.
# ---------------------------------------------------------
MODEL_NAME = "Qwen/Qwen2.5-VL-3B-Instruct"


# ---------------------------------------------------------
# 현재 실행 환경에 맞는 Device를 선택한다.
# 우선순위는 CUDA → MPS → CPU이다.
# ---------------------------------------------------------
def get_device():

    # -----------------------------------------------------
    # NVIDIA CUDA GPU가 있으면 CUDA를 사용한다.
    # -----------------------------------------------------
    if torch.cuda.is_available():
        return torch.device("cuda")

    # -----------------------------------------------------
    # Apple Silicon에서 MPS를 사용할 수 있으면 MPS를 사용한다.
    # -----------------------------------------------------
    if (
        hasattr(torch.backends, "mps")
        and torch.backends.mps.is_available()
    ):
        return torch.device("mps")

    # -----------------------------------------------------
    # GPU 가속 장치가 없으면 CPU를 사용한다.
    # -----------------------------------------------------
    return torch.device("cpu")


# ---------------------------------------------------------
# 실제 사용할 Device를 결정한다.
# ---------------------------------------------------------
device = get_device()


# ---------------------------------------------------------
# Device에 따라 기본 생성 Token 수를 결정한다.
# CPU에서는 생성 시간을 줄이기 위해 32를 사용한다.
# GPU에서는 좀 더 긴 답변을 위해 128을 사용한다.
# ---------------------------------------------------------
if device.type == "cpu":
    DEFAULT_MAX_NEW_TOKENS = 32
else:
    DEFAULT_MAX_NEW_TOKENS = 128


# ---------------------------------------------------------
# Device에 따라 Processor의 최대 이미지 Pixel 수를 결정한다.
#
# 이미지가 커질수록 Visual Token이 증가하여
# Vision Encoder와 LLM의 연산량이 증가한다.
#
# CPU에서는 속도를 위해 비교적 작은 이미지를 사용한다.
# GPU에서는 더 많은 이미지 정보를 사용할 수 있도록 크게 설정한다.
# ---------------------------------------------------------
if device.type == "cpu":
    MAX_PIXELS = 448 * 448
else:
    MAX_PIXELS = 1024 * 1024


# ---------------------------------------------------------
# Streamlit rerun마다 대형 모델을 다시 로드하지 않도록
# Processor와 Model을 Resource Cache에 저장한다.
# ---------------------------------------------------------
@st.cache_resource(show_spinner="Qwen2.5-VL 모델을 불러오는 중입니다...")
def load_model():

    # -----------------------------------------------------
    # CUDA에서는 bfloat16을 사용한다.
    # GPU 메모리 사용량을 줄이고 연산 효율을 높일 수 있다.
    # -----------------------------------------------------
    if device.type == "cuda":
        model_dtype = torch.bfloat16

    # -----------------------------------------------------
    # MPS에서는 float16을 사용한다.
    # -----------------------------------------------------
    elif device.type == "mps":
        model_dtype = torch.float16

    # -----------------------------------------------------
    # CPU에서는 호환성을 우선하여 float32를 사용한다.
    # -----------------------------------------------------
    else:
        model_dtype = torch.float32

    # -----------------------------------------------------
    # Qwen2.5-VL Processor를 불러온다.
    #
    # max_pixels를 제한하면 지나치게 큰 이미지가 입력되었을 때
    # 생성되는 Visual Token 수를 제한할 수 있다.
    # -----------------------------------------------------
    processor = AutoProcessor.from_pretrained(
        MODEL_NAME,
        max_pixels=MAX_PIXELS
    )

    # -----------------------------------------------------
    # Qwen2.5-VL 모델을 불러온다.
    # -----------------------------------------------------
    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        MODEL_NAME,
        torch_dtype=model_dtype,
        low_cpu_mem_usage=True
    )

    # -----------------------------------------------------
    # 모델을 선택한 Device로 이동한다.
    # -----------------------------------------------------
    model = model.to(device)

    # -----------------------------------------------------
    # Dropout 등 학습 전용 동작을 비활성화한다.
    # -----------------------------------------------------
    model.eval()

    # -----------------------------------------------------
    # Processor와 Model을 반환한다.
    # -----------------------------------------------------
    return processor, model


# ---------------------------------------------------------
# 업로드 파일의 내용을 이용하여 고유 Signature를 생성한다.
# 이미지 변경 여부를 확인하는 데 사용한다.
# ---------------------------------------------------------
def make_upload_signature(uploaded_files):

    # -----------------------------------------------------
    # 업로드된 파일이 없으면 빈 문자열을 반환한다.
    # -----------------------------------------------------
    if not uploaded_files:
        return ""

    # -----------------------------------------------------
    # 파일별 Hash를 저장할 리스트를 생성한다.
    # -----------------------------------------------------
    hashes = []

    # -----------------------------------------------------
    # 업로드 파일을 하나씩 처리한다.
    # -----------------------------------------------------
    for uploaded_file in uploaded_files:

        # -------------------------------------------------
        # 업로드 파일의 원본 Byte를 가져온다.
        # -------------------------------------------------
        file_bytes = uploaded_file.getvalue()

        # -------------------------------------------------
        # SHA-256 Hash를 생성한다.
        # -------------------------------------------------
        file_hash = hashlib.sha256(file_bytes).hexdigest()

        # -------------------------------------------------
        # 파일 이름과 Hash를 결합한다.
        # -------------------------------------------------
        hashes.append(
            f"{uploaded_file.name}:{file_hash}"
        )

    # -----------------------------------------------------
    # 모든 파일 Signature를 하나의 문자열로 반환한다.
    # -----------------------------------------------------
    return "|".join(hashes)


# ---------------------------------------------------------
# CPU에서 사용할 이미지의 크기를 줄이는 함수를 정의한다.
# ---------------------------------------------------------
def resize_image_for_device(image):

    # -----------------------------------------------------
    # CPU가 아니면 원본 크기를 유지한다.
    # Processor가 MAX_PIXELS에 맞추어 추가 처리한다.
    # -----------------------------------------------------
    if device.type != "cpu":
        return image

    # -----------------------------------------------------
    # CPU에서는 가장 긴 변의 최대 크기를 448 Pixel로 제한한다.
    # -----------------------------------------------------
    max_side = 448

    # -----------------------------------------------------
    # 이미지의 현재 크기를 가져온다.
    # -----------------------------------------------------
    width, height = image.size

    # -----------------------------------------------------
    # 이미 충분히 작으면 크기를 변경하지 않는다.
    # -----------------------------------------------------
    if max(width, height) <= max_side:
        return image

    # -----------------------------------------------------
    # 원본 가로세로 비율을 유지하면서 축소 비율을 계산한다.
    # -----------------------------------------------------
    scale = max_side / max(width, height)

    # -----------------------------------------------------
    # 새로운 가로 크기를 계산한다.
    # -----------------------------------------------------
    new_width = int(width * scale)

    # -----------------------------------------------------
    # 새로운 세로 크기를 계산한다.
    # -----------------------------------------------------
    new_height = int(height * scale)

    # -----------------------------------------------------
    # 이미지를 축소한다.
    # -----------------------------------------------------
    resized_image = image.resize(
        (new_width, new_height),
        Image.Resampling.LANCZOS
    )

    # -----------------------------------------------------
    # 축소한 이미지를 반환한다.
    # -----------------------------------------------------
    return resized_image


# ---------------------------------------------------------
# 업로드 파일을 RGB PIL Image 리스트로 변환한다.
# ---------------------------------------------------------
def load_uploaded_images(uploaded_files):

    # -----------------------------------------------------
    # 변환된 이미지를 저장할 리스트를 만든다.
    # -----------------------------------------------------
    images = []

    # -----------------------------------------------------
    # 업로드 파일을 하나씩 처리한다.
    # -----------------------------------------------------
    for uploaded_file in uploaded_files:

        # -------------------------------------------------
        # 업로드 파일을 PIL Image로 연다.
        # -------------------------------------------------
        image = Image.open(uploaded_file)

        # -------------------------------------------------
        # 모델 입력 형식을 일관되게 만들기 위해 RGB로 변환한다.
        # -------------------------------------------------
        image = image.convert("RGB")

        # -------------------------------------------------
        # 현재 Device에 적합한 이미지 크기로 조정한다.
        # -------------------------------------------------
        image = resize_image_for_device(image)

        # -------------------------------------------------
        # 처리한 이미지를 리스트에 추가한다.
        # -------------------------------------------------
        images.append(image)

    # -----------------------------------------------------
    # 이미지 리스트를 반환한다.
    # -----------------------------------------------------
    return images


# ---------------------------------------------------------
# Streamlit Session State의 기본 변수를 초기화한다.
# ---------------------------------------------------------
def initialize_session_state():

    # -----------------------------------------------------
    # 대화 History를 저장할 리스트를 초기화한다.
    # -----------------------------------------------------
    if "messages" not in st.session_state:
        st.session_state.messages = []

    # -----------------------------------------------------
    # 업로드 이미지를 저장할 리스트를 초기화한다.
    # -----------------------------------------------------
    if "images" not in st.session_state:
        st.session_state.images = []

    # -----------------------------------------------------
    # 이미지 Signature를 초기화한다.
    # -----------------------------------------------------
    if "image_signature" not in st.session_state:
        st.session_state.image_signature = ""


# ---------------------------------------------------------
# 현재 대화를 초기화한다.
# ---------------------------------------------------------
def reset_conversation():

    # -----------------------------------------------------
    # 저장된 모든 대화 내용을 제거한다.
    # -----------------------------------------------------
    st.session_state.messages = []


# ---------------------------------------------------------
# Qwen2.5-VL에 전달할 Multimodal Message를 생성한다.
# ---------------------------------------------------------
def build_model_messages(
    images,
    chat_history,
    current_prompt
):

    # -----------------------------------------------------
    # 모델에 전달할 Message 리스트를 만든다.
    # -----------------------------------------------------
    model_messages = []

    # -----------------------------------------------------
    # 첫 번째 질문이면 이미지와 질문을 하나의 User Message에 넣는다.
    # -----------------------------------------------------
    if not chat_history:

        # -------------------------------------------------
        # 첫 User Message의 Content를 만든다.
        # -------------------------------------------------
        first_content = []

        # -------------------------------------------------
        # 모든 이미지를 Message에 추가한다.
        # -------------------------------------------------
        for image in images:
            first_content.append(
                {
                    "type": "image",
                    "image": image
                }
            )

        # -------------------------------------------------
        # 사용자 질문을 이미지 다음에 추가한다.
        # -------------------------------------------------
        first_content.append(
            {
                "type": "text",
                "text": current_prompt
            }
        )

        # -------------------------------------------------
        # 완성된 첫 User Message를 추가한다.
        # -------------------------------------------------
        model_messages.append(
            {
                "role": "user",
                "content": first_content
            }
        )

        # -------------------------------------------------
        # 첫 질문용 Message를 반환한다.
        # -------------------------------------------------
        return model_messages

    # -----------------------------------------------------
    # 첫 번째 과거 User Message를 가져온다.
    # -----------------------------------------------------
    first_history_message = chat_history[0]

    # -----------------------------------------------------
    # 첫 User Message의 Content를 다시 만든다.
    # -----------------------------------------------------
    first_content = []

    # -----------------------------------------------------
    # 첫 Message에 현재 Session 이미지를 다시 포함한다.
    # -----------------------------------------------------
    for image in images:
        first_content.append(
            {
                "type": "image",
                "image": image
            }
        )

    # -----------------------------------------------------
    # 첫 번째 사용자 질문을 추가한다.
    # -----------------------------------------------------
    first_content.append(
        {
            "type": "text",
            "text": first_history_message["content"]
        }
    )

    # -----------------------------------------------------
    # 첫 User Message를 모델 Message에 추가한다.
    # -----------------------------------------------------
    model_messages.append(
        {
            "role": "user",
            "content": first_content
        }
    )

    # -----------------------------------------------------
    # 첫 Message 이후의 대화 History를 복원한다.
    # -----------------------------------------------------
    for message in chat_history[1:]:

        # -------------------------------------------------
        # User Message를 모델 형식으로 변환한다.
        # -------------------------------------------------
        if message["role"] == "user":
            model_messages.append(
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "text",
                            "text": message["content"]
                        }
                    ]
                }
            )

        # -------------------------------------------------
        # Assistant Message를 모델 형식으로 추가한다.
        # -------------------------------------------------
        else:
            model_messages.append(
                {
                    "role": "assistant",
                    "content": message["content"]
                }
            )

    # -----------------------------------------------------
    # 현재 새 질문을 마지막에 추가한다.
    # -----------------------------------------------------
    model_messages.append(
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": current_prompt
                }
            ]
        }
    )

    # -----------------------------------------------------
    # 완성된 Message History를 반환한다.
    # -----------------------------------------------------
    return model_messages


# ---------------------------------------------------------
# Qwen2.5-VL을 이용하여 응답을 생성한다.
# ---------------------------------------------------------
def generate_answer(
    images,
    chat_history,
    prompt,
    processor,
    model,
    max_new_tokens
):

    # -----------------------------------------------------
    # 전체 추론 시간을 측정하기 위해 시작 시간을 기록한다.
    # -----------------------------------------------------
    start_time = time.perf_counter()

    # -----------------------------------------------------
    # UI History를 Qwen2.5-VL용 Multimodal Message로 변환한다.
    # -----------------------------------------------------
    model_messages = build_model_messages(
        images,
        chat_history,
        prompt
    )

    # -----------------------------------------------------
    # Qwen2.5-VL Chat Template을 적용한다.
    #
    # 이미지와 텍스트를 모델이 처리할 수 있는 Tensor로 변환한다.
    # -----------------------------------------------------
    inputs = processor.apply_chat_template(
        model_messages,
        tokenize=True,
        add_generation_prompt=True,
        add_vision_id=True,
        return_dict=True,
        return_tensors="pt"
    )

    # -----------------------------------------------------
    # 입력 Tensor를 모델과 동일한 Device로 이동한다.
    # -----------------------------------------------------
    inputs = inputs.to(device)

    # -----------------------------------------------------
    # Prompt 부분의 Token 길이를 저장한다.
    # -----------------------------------------------------
    input_lengths = [
        len(input_ids)
        for input_ids in inputs["input_ids"]
    ]

    # -----------------------------------------------------
    # 추론에서는 Gradient 계산이 필요하지 않으므로 비활성화한다.
    # -----------------------------------------------------
    with torch.inference_mode():

        # -------------------------------------------------
        # 모델이 새로운 응답 Token을 생성한다.
        #
        # CPU에서는 max_new_tokens를 작게 설정하는 것이 중요하다.
        # -------------------------------------------------
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False
        )

    # -----------------------------------------------------
    # 전체 결과에서 입력 Prompt Token을 제외한다.
    # -----------------------------------------------------
    response_ids = [
        output_ids[input_length:]
        for input_length, output_ids in zip(
            input_lengths,
            generated_ids
        )
    ]

    # -----------------------------------------------------
    # 생성된 Token을 자연어 문자열로 변환한다.
    # -----------------------------------------------------
    answer = processor.batch_decode(
        response_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    )[0]

    # -----------------------------------------------------
    # 추론 종료 시간을 기록한다.
    # -----------------------------------------------------
    end_time = time.perf_counter()

    # -----------------------------------------------------
    # 총 추론 시간을 계산한다.
    # -----------------------------------------------------
    elapsed_time = end_time - start_time

    # -----------------------------------------------------
    # 생성된 답변과 추론 시간을 함께 반환한다.
    # -----------------------------------------------------
    return answer, elapsed_time


# ---------------------------------------------------------
# Session State를 초기화한다.
# ---------------------------------------------------------
initialize_session_state()


# ---------------------------------------------------------
# Sidebar에 설정 영역을 만든다.
# ---------------------------------------------------------
st.sidebar.header("설정")


# ---------------------------------------------------------
# 현재 사용 중인 모델 이름을 표시한다.
# ---------------------------------------------------------
st.sidebar.write(
    "Model:",
    MODEL_NAME
)


# ---------------------------------------------------------
# 현재 선택된 Device를 표시한다.
# ---------------------------------------------------------
st.sidebar.write(
    "Device:",
    str(device)
)


# ---------------------------------------------------------
# 현재 Device에 따른 데이터 형식을 설명한다.
# ---------------------------------------------------------
if device.type == "cuda":
    st.sidebar.success(
        "CUDA GPU를 사용합니다."
    )
elif device.type == "mps":
    st.sidebar.success(
        "Apple MPS를 사용합니다."
    )
else:
    st.sidebar.warning(
        "CPU 모드입니다. Qwen2.5-VL-3B 추론은 느릴 수 있습니다."
    )


# ---------------------------------------------------------
# CPU와 GPU에 따라 적절한 Token 범위를 설정한다.
# ---------------------------------------------------------
if device.type == "cpu":

    # -----------------------------------------------------
    # CPU에서는 짧은 답변을 사용하도록 제한한다.
    # -----------------------------------------------------
    max_new_tokens = st.sidebar.slider(
        "최대 생성 Token",
        min_value=16,
        max_value=128,
        value=DEFAULT_MAX_NEW_TOKENS,
        step=16
    )

else:

    # -----------------------------------------------------
    # GPU에서는 좀 더 긴 답변을 생성할 수 있도록 한다.
    # -----------------------------------------------------
    max_new_tokens = st.sidebar.slider(
        "최대 생성 Token",
        min_value=32,
        max_value=512,
        value=DEFAULT_MAX_NEW_TOKENS,
        step=32
    )


# ---------------------------------------------------------
# 대화 초기화 버튼을 만든다.
# ---------------------------------------------------------
reset_button = st.sidebar.button(
    "대화 초기화"
)


# ---------------------------------------------------------
# 초기화 버튼이 눌리면 대화 History를 제거한다.
# ---------------------------------------------------------
if reset_button:

    # -----------------------------------------------------
    # 저장된 대화를 제거한다.
    # -----------------------------------------------------
    reset_conversation()

    # -----------------------------------------------------
    # Streamlit 화면을 다시 실행한다.
    # -----------------------------------------------------
    st.rerun()


# ---------------------------------------------------------
# 이미지 업로드 영역을 표시한다.
# ---------------------------------------------------------
st.subheader("이미지 업로드")


# ---------------------------------------------------------
# 하나 이상의 이미지를 업로드할 수 있도록 한다.
# ---------------------------------------------------------
uploaded_files = st.file_uploader(
    "분석할 이미지를 선택하세요.",
    type=[
        "png",
        "jpg",
        "jpeg",
        "webp"
    ],
    accept_multiple_files=True
)


# ---------------------------------------------------------
# 현재 업로드 파일의 Signature를 계산한다.
# ---------------------------------------------------------
current_signature = make_upload_signature(
    uploaded_files
)


# ---------------------------------------------------------
# 업로드 이미지가 이전 이미지와 다른지 확인한다.
# ---------------------------------------------------------
if (
    current_signature
    != st.session_state.image_signature
):

    # -----------------------------------------------------
    # 현재 Signature를 Session State에 저장한다.
    # -----------------------------------------------------
    st.session_state.image_signature = current_signature

    # -----------------------------------------------------
    # 업로드 이미지가 있으면 PIL Image로 변환한다.
    # -----------------------------------------------------
    if uploaded_files:
        st.session_state.images = load_uploaded_images(
            uploaded_files
        )

    # -----------------------------------------------------
    # 이미지가 없으면 Image List를 비운다.
    # -----------------------------------------------------
    else:
        st.session_state.images = []

    # -----------------------------------------------------
    # 이미지가 변경되면 이전 대화를 초기화한다.
    # -----------------------------------------------------
    reset_conversation()


# ---------------------------------------------------------
# CPU에서 여러 이미지가 입력되면 속도 경고를 표시한다.
# ---------------------------------------------------------
if (
    device.type == "cpu"
    and len(st.session_state.images) > 1
):
    st.warning(
        "CPU에서는 여러 이미지를 동시에 분석하면 매우 느릴 수 있습니다. "
        "먼저 이미지 1장으로 테스트하는 것을 권장합니다."
    )


# ---------------------------------------------------------
# 업로드된 이미지가 있으면 미리보기를 표시한다.
# ---------------------------------------------------------
if st.session_state.images:

    # -----------------------------------------------------
    # 최대 4개의 Column을 생성한다.
    # -----------------------------------------------------
    columns = st.columns(
        min(
            len(st.session_state.images),
            4
        )
    )

    # -----------------------------------------------------
    # 업로드 이미지를 순서대로 표시한다.
    # -----------------------------------------------------
    for index, image in enumerate(
        st.session_state.images
    ):

        # -------------------------------------------------
        # 이미지를 표시할 Column을 선택한다.
        # -------------------------------------------------
        column = columns[
            index % len(columns)
        ]

        # -------------------------------------------------
        # 선택한 Column에 이미지를 표시한다.
        # -------------------------------------------------
        with column:
            st.image(
                image,
                caption=(
                    f"Picture {index + 1} "
                    f"({image.width}×{image.height})"
                ),
                width="stretch"
            )

# ---------------------------------------------------------
# 이미지가 없으면 안내 메시지를 표시한다.
# ---------------------------------------------------------
else:
    st.info(
        "먼저 이미지를 한 장 이상 업로드하세요."
    )


# ---------------------------------------------------------
# 기존 대화 History를 화면에 표시한다.
# ---------------------------------------------------------
for message in st.session_state.messages:

    # -----------------------------------------------------
    # Message Role에 맞는 Chat 영역을 만든다.
    # -----------------------------------------------------
    with st.chat_message(
        message["role"]
    ):

        # -------------------------------------------------
        # Message 내용을 출력한다.
        # -------------------------------------------------
        st.markdown(
            message["content"]
        )


# ---------------------------------------------------------
# 사용자의 질문을 입력받는다.
# ---------------------------------------------------------
prompt = st.chat_input(
    "이미지에 대해 한국어로 질문하세요."
)


# ---------------------------------------------------------
# 사용자가 질문을 입력한 경우에만 처리한다.
# ---------------------------------------------------------
if prompt:

    # -----------------------------------------------------
    # 이미지가 없으면 모델을 실행하지 않는다.
    # -----------------------------------------------------
    if not st.session_state.images:
        st.warning(
            "질문하기 전에 이미지를 업로드하세요."
        )

    # -----------------------------------------------------
    # 이미지가 있으면 멀티모달 추론을 수행한다.
    # -----------------------------------------------------
    else:

        # -------------------------------------------------
        # 현재 질문이 추가되기 전의 History를 복사한다.
        # -------------------------------------------------
        history_before_prompt = list(
            st.session_state.messages
        )

        # -------------------------------------------------
        # 사용자 질문을 화면에 표시한다.
        # -------------------------------------------------
        with st.chat_message("user"):
            st.markdown(prompt)

        # -------------------------------------------------
        # 사용자 질문을 Session History에 저장한다.
        # -------------------------------------------------
        st.session_state.messages.append(
            {
                "role": "user",
                "content": prompt
            }
        )

        # -------------------------------------------------
        # 최초 추론 시 모델을 로드한다.
        # 이후에는 Streamlit Resource Cache에서 가져온다.
        # -------------------------------------------------
        processor, model = load_model()

        # -------------------------------------------------
        # Assistant 응답 영역을 생성한다.
        # -------------------------------------------------
        with st.chat_message("assistant"):

            # -------------------------------------------------
            # 추론 중 진행 상태를 표시한다.
            # -------------------------------------------------
            with st.spinner(
                "이미지를 분석하고 있습니다..."
            ):

                # ---------------------------------------------
                # 모델 추론 중 발생할 수 있는 오류를 처리한다.
                # ---------------------------------------------
                try:

                    # -----------------------------------------
                    # Qwen2.5-VL을 실행하여 답변을 생성한다.
                    # -----------------------------------------
                    answer, elapsed_time = generate_answer(
                        st.session_state.images,
                        history_before_prompt,
                        prompt,
                        processor,
                        model,
                        max_new_tokens
                    )

                # ---------------------------------------------
                # 추론 중 오류가 발생하면 내용을 화면에 표시한다.
                # ---------------------------------------------
                except Exception as error:

                    # -----------------------------------------
                    # 오류 내용을 문자열로 만든다.
                    # -----------------------------------------
                    answer = (
                        "모델 추론 중 오류가 발생했습니다.\n\n"
                        f"`{type(error).__name__}: {error}`"
                    )

                    # -----------------------------------------
                    # 오류가 발생한 경우 추론 시간을 초기화한다.
                    # -----------------------------------------
                    elapsed_time = None

                    # -----------------------------------------
                    # CUDA 환경이면 사용하지 않는 Cache를 정리한다.
                    # -----------------------------------------
                    if device.type == "cuda":
                        torch.cuda.empty_cache()

                    # -----------------------------------------
                    # Python Garbage Collection을 실행한다.
                    # -----------------------------------------
                    gc.collect()

            # -------------------------------------------------
            # 생성된 답변을 출력한다.
            # -------------------------------------------------
            st.markdown(answer)

            # -------------------------------------------------
            # 정상 추론이면 소요 시간을 표시한다.
            # -------------------------------------------------
            if elapsed_time is not None:
                st.caption(
                    f"추론 시간: {elapsed_time:.2f}초 | "
                    f"Device: {device.type} | "
                    f"최대 생성 Token: {max_new_tokens}"
                )

        # -------------------------------------------------
        # Assistant 응답을 Session History에 저장한다.
        # -------------------------------------------------
        st.session_state.messages.append(
            {
                "role": "assistant",
                "content": answer
            }
        )

        
print(
    "app.py source characters : ",
    len(APP_SOURCE
    )
)


2026-10-08 11:20:34.299 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-08 11:20:34.301 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-08 11:20:34.302 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-08 11:20:34.304 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-08 11:20:34.305 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-08 11:20:34.306 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-08 11:20:34.307 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-08 11:20:34.310 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

app.py source characters :  0


# 6. `app.py` Python 문법 검사

파일로 저장하기 전에 `compile()`로 Python 문법을 검사한다.

In [9]:
# ---------------------------------------------------------
# Streamlit 전체 소스의 Python 문법을 검사한다.
# 오류가 있으면 이 셀에서 SyntaxError가 발생한다.
# ---------------------------------------------------------
compile(
    APP_SOURCE,
    "app.py",
    "exec"
)

# ---------------------------------------------------------
# 문법 검사가 완료되었음을 출력한다.
# ---------------------------------------------------------
print(
    "app.py syntax check: OK"
)

app.py syntax check: OK


# 7. `app.py` 파일 생성

In [10]:
# ---------------------------------------------------------
# Streamlit 전체 소스를 UTF-8 형식으로 app.py에 저장한다.
# ---------------------------------------------------------
APP_PATH.write_text(
    APP_SOURCE,
    encoding="utf-8"
)

# ---------------------------------------------------------
# 생성된 app.py 파일의 절대 경로를 출력한다.
# ---------------------------------------------------------
print(
    "Created:",
    APP_PATH.resolve()
)

Created: C:\sk-encoa\runpod\qwen_vl_streamlit\app.py


# 8. 실행 안내 파일 작성

In [11]:
# ---------------------------------------------------------
# 프로젝트 실행 방법을 안내하는 문자열을 정의한다.
# ---------------------------------------------------------
README_TEXT = """Qwen2.5-VL Streamlit Multimodal Chat

1. 패키지 설치

uv add "torch==2.8.0" "torchvision==0.23.0" "transformers==4.57.2" "accelerate>=1.5,<2" "pillow>=10,<12" "streamlit>=1.40,<2"

2. 실행

uv run streamlit run qwen_vl_streamlit/app.py

3. 기능

- 이미지 한 장 또는 여러 장 업로드
- 한국어 질문
- 멀티이미지 비교
- 멀티턴 대화
- 대화 초기화
- CUDA / MPS / CPU 선택

4. 모델

Qwen/Qwen2.5-VL-3B-Instruct

주의:
최초 실행 시 사전학습 모델 가중치 다운로드가 필요합니다.
CPU에서는 3B급 멀티모달 모델 추론이 매우 느릴 수 있습니다.
"""

# ---------------------------------------------------------
# 실행 안내 문자열을 README.txt 파일로 저장한다.
# ---------------------------------------------------------
README_PATH.write_text(
    README_TEXT,
    encoding="utf-8"
)

# ---------------------------------------------------------
# 생성된 안내 파일 경로를 출력한다.
# ---------------------------------------------------------
print(
    "Created:",
    README_PATH.resolve()
)

Created: C:\sk-encoa\runpod\qwen_vl_streamlit\README.txt


# 9. 생성된 프로젝트 파일 확인

In [12]:
# ---------------------------------------------------------
# 프로젝트 폴더 안의 파일을 이름순으로 가져온다.
# ---------------------------------------------------------
project_files = sorted(
    PROJECT_DIR.iterdir()
)

# ---------------------------------------------------------
# 생성된 파일 이름을 하나씩 출력한다.
# ---------------------------------------------------------
for file_path in project_files:
    print(
        file_path.name
    )

app.py
README.txt


# 10. 애플리케이션 핵심 구조

웹 애플리케이션의 실행 흐름은 다음과 같다.

```text
Browser
   ↓
st.file_uploader()
   ↓
PIL Image List
   ↓
st.session_state.images
   ↓
st.chat_input()
   ↓
history_before_prompt
   ↓
build_model_messages()
   ↓
processor.apply_chat_template()
   ↓
pixel_values + input_ids + ...
   ↓
Qwen2.5-VL
   ↓
model.generate()
   ↓
입력 Token 제거
   ↓
batch_decode()
   ↓
Assistant Response
   ↓
st.session_state.messages
```

# 11. 왜 Session State가 필요한가

Streamlit은 Widget 입력이 발생할 때 Script를 다시 실행한다.

다음 일반 변수는 rerun 시 새로 생성된다.

```python
messages = []
```

반면 다음 상태는 사용자 Session 동안 유지할 수 있다.

```python
st.session_state.messages
```

따라서 대화형 UI에서는 Session State가 핵심이다.

# 12. UI History와 Model History를 분리한 이유

화면 표시용 History에는 문자열만 저장한다.

```python
{
    "role": "user",
    "content": "그 객체의 색은?"
}
```

반면 Qwen2.5-VL에 전달할 첫 User Message에는 실제 이미지가 필요하다.

```text
User
├── Image 1
├── Image 2
└── 첫 질문
```

따라서 `build_model_messages()`가 화면용 History와 현재 PIL Image를 결합하여 모델용 Multimodal History를 다시 만든다.

이렇게 하면 Session State 구조가 단순해지고 UI 출력과 모델 입력의 역할을 분리할 수 있다.

# 13. 첫 질문과 후속 질문의 차이

첫 질문에서는 이미지와 질문을 함께 전달한다.

```text
User
├── Picture 1
├── Picture 2
└── 두 이미지를 비교해줘.
```

두 번째 질문부터는 이전 History를 유지하면서 텍스트만 추가한다.

```text
Assistant
└── 첫 번째는 강아지이고 두 번째는 자동차입니다.

User
└── 첫 번째 이미지에 대해서만 더 설명해줘.
```

모델 입력에는 첫 Turn의 이미지가 여전히 포함되어 있으므로 후속 질문에서도 이미지 문맥을 참조할 수 있다.

# 14. 이미지가 변경되면 대화를 초기화하는 이유

이전 이미지를 대상으로 다음 대화를 진행했다고 가정한다.

```text
Image A
User: 이 동물은 무엇이야?
Assistant: 강아지입니다.
```

사용자가 새로운 Image B로 변경했는데 이전 History가 그대로 남아 있으면 `"이 동물"`이 어느 이미지를 의미하는지 혼동할 수 있다.

따라서 업로드 파일의 Hash를 계산하여 이미지가 변경되면 대화를 초기화한다.

```text
업로드 이미지 변경
      ↓
SHA-256 Signature 변경
      ↓
images 교체
      ↓
messages 초기화
```

# 15. 여러 이미지 비교 사용 예

이미지 2장을 업로드한 후 다음과 같이 질문할 수 있다.

```text
두 이미지의 주요 객체를 각각 알려줘.

첫 번째 이미지와 두 번째 이미지의 공통점은?

두 이미지의 가장 큰 차이는?

첫 번째 이미지에만 있는 특징을 알려줘.

두 번째 이미지에 대해서만 더 자세히 설명해줘.
```

`add_vision_id=True`를 사용하므로 여러 시각 입력을 참조하는 Prompt를 구성할 때 이미지 구분이 쉬워진다.

# 16. 메모리 관리

3B급 VLM은 일반적인 Text Model보다 이미지 처리까지 수행하므로 메모리 사용량이 크다.

이번 앱은 다음 원칙을 사용한다.

```text
모델 한 번 로드
    ↓
@st.cache_resource
    ↓
rerun에서 재사용
```

추론 오류가 발생하면 CUDA Cache와 Python Garbage Collection을 정리한다.

실습 GPU 메모리가 부족하다면 다음을 우선 조정한다.

```text
업로드 이미지 수 감소
이미지 해상도 감소
max_new_tokens 감소
더 작은 Batch 사용
한 번에 하나의 사용자 Session 사용
```

# 17. 실행 방법

Notebook에서 `app.py`가 생성된 후 Terminal에서 실행한다.

```bash
uv run streamlit run qwen_vl_streamlit/app.py
```

브라우저가 열리면 다음 순서로 실습한다.

```text
1. 이미지 업로드
2. 이미지 확인
3. 첫 한국어 질문
4. 모델 답변 확인
5. 후속 질문
6. 여러 이미지 비교
7. 대화 초기화
```

# 18. 실습 시나리오 1 - 단일 이미지

이미지 한 장을 업로드한다.

질문 예:

```text
이 이미지의 주요 객체는 무엇이야?

그 객체의 색이나 외형을 설명해줘.

그 객체는 무엇을 하고 있어?

배경은 어떻게 보여?

지금까지 내용을 한 문장으로 정리해줘.
```

두 번째 질문부터는 이전 문맥을 유지하는지 확인한다.

# 19. 실습 시나리오 2 - 멀티이미지

이미지 두 장 이상을 업로드한다.

질문 예:

```text
각 이미지의 주요 객체를 알려줘.

Picture 1과 Picture 2의 공통점은?

두 이미지의 차이점을 설명해줘.

Picture 1에 대해서만 자세히 설명해줘.

Picture 2의 객체는 어떤 용도로 사용돼?
```

여러 이미지 참조가 정확하게 유지되는지 확인한다.

# 20. 실습 시나리오 3 - 오류 분석

다음 오류를 찾아 기록한다.

```text
Object Error
Attribute Error
Count Error
Image Confusion
Reference Resolution Error
Context Consistency Error
Hallucination
```

예:

```text
질문:
Picture 1의 객체는 무엇인가?

실제:
dog

모델:
cat

오류:
Object Error
```

또는

```text
질문:
두 번째 이미지의 색은?

모델:
첫 번째 이미지의 색을 설명함

오류:
Image Confusion
```

# 21. 확인 문제

1. Streamlit에서 Session State를 사용하는 이유는 무엇인가?
2. `@st.cache_resource`는 이번 앱에서 어떤 역할을 하는가?
3. `st.file_uploader(..., accept_multiple_files=True)`는 무엇을 가능하게 하는가?
4. 업로드 이미지를 RGB로 변환하는 이유는 무엇인가?
5. `st.chat_input()`은 어떤 역할을 하는가?
6. 첫 번째 User Message에 이미지를 함께 넣는 이유는 무엇인가?
7. 후속 질문에서는 이미지를 다시 업로드해야 하는가?
8. `build_model_messages()`는 어떤 역할을 하는가?
9. `add_vision_id=True`는 어떤 상황에서 특히 유용한가?
10. `torch.inference_mode()`를 사용하는 이유는 무엇인가?
11. `input_lengths`가 필요한 이유는 무엇인가?
12. 이미지가 변경되면 기존 대화를 초기화하는 이유는 무엇인가?
13. SHA-256 Signature는 이번 앱에서 무엇을 판단하는 데 사용하는가?
14. `max_new_tokens`를 줄이면 어떤 자원을 절약할 수 있는가?
15. Streamlit rerun과 일반 Python Script 실행의 차이를 설명해보라.

# 22. 확인 문제 정답

1. Widget 조작으로 Script가 다시 실행되어도 사용자 Session의 대화와 이미지 상태를 유지하기 위해서이다.
2. 대형 Qwen2.5-VL 모델과 Processor를 rerun마다 다시 로드하지 않고 재사용한다.
3. 한 번에 여러 이미지 파일을 업로드할 수 있게 한다.
4. 입력 이미지 Channel 형식을 일관되게 만들기 위해서이다.
5. Chat 형태로 사용자의 자연어 질문을 입력받는다.
6. VLM이 첫 질문과 시각 정보를 함께 조건으로 사용하도록 하기 위해서이다.
7. 아니다. 현재 구현에서는 첫 Turn의 이미지를 Session State에 보관하고 모델 History를 만들 때 다시 포함한다.
8. UI용 문자열 History와 PIL Image를 결합하여 Qwen2.5-VL에 전달할 Multimodal Message History를 만든다.
9. 여러 이미지 또는 비디오가 포함되어 각 시각 입력을 구분하여 참조해야 할 때 유용하다.
10. 추론 시 Gradient 계산을 제거하여 불필요한 연산과 메모리 사용을 줄이기 위해서이다.
11. `generate()` 결과에서 입력 Prompt Token을 제외하고 새 Assistant 응답만 추출하기 위해서이다.
12. 이전 이미지 문맥과 새 이미지 문맥이 섞이는 것을 방지하기 위해서이다.
13. 사용자가 업로드한 파일 집합이 이전과 달라졌는지 확인하는 데 사용한다.
14. 생성 연산 시간과 생성 과정의 메모리 사용을 줄일 수 있다.
15. Streamlit은 Widget 상호작용에 따라 Script가 위에서 아래로 다시 실행되므로 상태를 별도로 관리해야 한다.

# 23. 실습 과제

## 과제 1. 단일 이미지 멀티턴

이미지 한 장을 업로드하고 5회 이상 후속 질문을 수행한다.

평가 항목:

```text
Object Accuracy
Attribute Accuracy
Context Consistency
Reference Resolution
Hallucination
```

## 과제 2. 이미지 3장 비교

서로 다른 이미지 3장을 업로드한다.

다음 질문을 수행한다.

```text
각 이미지의 객체는?
Picture 1과 Picture 2의 공통점은?
Picture 2와 Picture 3의 차이점은?
Picture 1에만 있는 특징은?
세 이미지를 표 형식으로 비교해줘.
```

## 과제 3. 오류 분석

최소 3개 오류 사례를 저장한다.

```text
질문
모델 답변
실제 이미지 내용
오류 유형
오류 원인
개선 Prompt
```

## 과제 4. UI 확장

다음 중 두 가지 이상을 추가한다.

```text
대화 다운로드
답변 복사
이미지별 이름 지정
System Prompt 입력
Temperature 설정
대화 평가 점수
오류 유형 선택
JSON 결과 저장
```

# 24. 전체 과정 연결

```text
Multimodal Concept
        ↓
Embedding
        ↓
Shared Embedding Space
        ↓
Contrastive Learning
        ↓
CLIP
├── Zero-shot Classification
└── Image Retrieval
        ↓
BLIP
├── Captioning
└── VQA
        ↓
CLIP + BLIP Project
        ↓
Qwen2.5-VL
├── Korean VQA
├── Multi-Image
└── Multi-turn
        ↓
Streamlit Application
├── Image Upload
├── Korean Chat
├── Session State
├── Multi-Image Comparison
└── Context-aware Conversation
```

이번 단계부터 모델 실습이 실제 사용자가 조작할 수 있는 애플리케이션으로 확장된다.

# 25. 다음 단계

다음 단계에서는 현재 Streamlit 앱을 더 실제 서비스에 가깝게 확장할 수 있다.

```text
Streamlit Multimodal Chat
        ↓
대화 결과 저장
        ↓
JSON / CSV Export
        ↓
평가 Dashboard
        ↓
질문별 점수
        ↓
오류 유형 통계
        ↓
VLM 성능 평가
```

다음 Notebook은 **`13_VLM_성능평가_대시보드.ipynb`**로 확장할 수 있다.